In [1]:
# import scanpy as sc
# import pandas as pd
# adata =sc.read_h5ad("/mnt/d/ST_Graduation_Project_data/DATA18/scRNA.h5ad")
# adata.obs['cell_type']=adata.obs['celltype_final']
# adata.write_h5ad("/mnt/d/ST_Graduation_Project_data/DATA18/scRNA.h5ad")
# adata_st = sc.read_h5ad("/mnt/d/ST_Graduation_Project_data/DATA18/Spatial.h5ad")
# st_label = adata_st.uns['density']
# st_label_df = pd.DataFrame(st_label)
# st_label_df.to_csv("/mnt/d/ST_Graduation_Project/SC_MAP_ST/evaluate/Data18_density.csv")

In [2]:
%run ../stage1.py \
    --sc_file "/mnt/d/ST_Graduation_Project_data/DATA18/scRNA.h5ad" \
    --st_file "/mnt/d/ST_Graduation_Project_data/DATA18/Spatial.h5ad" \
    --n_epochs 100 \
    --resolution 4 \
    --top_n_per_type 100 \
    --latent_dim 256 \
    --output_dir ../deconv_results/DATA18 \
    --marker_selection_method l1 
#    --precomputed_marker_file "../deconv_results/DATA18_backup/final_genes.txt"

/home/mwc/miniconda3/envs/dl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
Stage 1 Training: VAE (SC + ST, Marker Genes)
Configuration:
   Marker genes per type: 100
   Leiden resolution: 4.0
   Batch size: 256
   Epochs: 100
   Learning rate: 0.0005
   Beta (KL weight): 0.1
   Hidden dims: [512, 256]
   Latent dim: 256
   Loss type: ZINB
   Lambda MMD: 1.0
   Dual Decoder: True
   Aggregation method: mean
Loading datasets...
   Loading SC: /mnt/d/ST_Graduation_Project_data/DATA18/scRNA.h5ad
   SC shape: (9999, 24683)
   SC avg counts/cell: 11809.2 (from X)
   Loading ST: /mnt/d/ST_Graduation_Project_data/DATA18/Spatial.h5ad
   ST shape: (1000, 43324)
   Common genes: 19886
   SC final: (9999, 19886)
   ST final: (1000, 19886)
Computing clusters and marker genes...
Starting clustering analysis...
Clustering results: 74 clusters
Marker genes per cluster:
   0: 100 -> 100 (after L1)
   1: 100 -> 100 (after L1)
   10: 100 -> 100 (after L1)
   11: 100 -> 100 (after L1)
   12: 100 -> 100 (after L1)
   13: 100 -> 100 (after L1)
   14: 100 -> 100 (

VAE Training: 100%|██████████| 100/100 [03:17<00:00,  1.98s/epoch, Train=2127.0230, Recon=2118.9287, KL=80.9425, MMD=0.0219, Test=2139.7088]


📊 Computing cluster centers using ALL SC data (train + test)...
   ALL SC data: (9999, 1655)
   Number of clusters: 74
   Computed embeddings shape: (9999, 256)
Computing cluster centers with mean aggregation...
   Completed: 74 clusters with mean centers and expressions
Extracting celltype-cluster mapping (using 'cell_type' column)...
Visualizing modality alignment...
Generating UMAP visualization for modality alignment...
   Computing UMAP on 10889 samples with 256 dims...
   UMAP visualization saved to: ../deconv_results/DATA18/modality_alignment_umap.png
Saving model to: ../deconv_results/DATA18/final_vae.pth
   Average cell counts: 11809.2 (for Stage 2 scale factor)
✅ Saved VAE model (weights only): ../deconv_results/DATA18/final_vae.pth
Saving cluster data to: ../deconv_results/DATA18/final_vae_cluster_data.npz
   ✓ Cluster IDs: 74
   ✓ Prototypes: (74, 256)
   ✓ Expressions (marker): (74, 1655)
   ✓ Expressions (full): 74 clusters × 19886 genes
   ✓ Celltype mapping: 74 clusters

In [ ]:
%run ../stage2.py \
    --st_file "/mnt/d/ST_Graduation_Project_data/DATA18/Spatial.h5ad" \
    --stage1_model_path "../deconv_results/DATA18/final_vae.pth" \
    --output_dir "../deconv_results/DATA18/" \
    --gat_hidden_dim 512 \
    --gat_layers 4 \
    --lr 5e-3 \
    --k_spatial 10 \
    --k_celltype 15 \
    --batch_size 512 \
    --n_epochs 2501

Sample name: Spatial
Stage 1 model: ../deconv_results/DATA18/final_vae.pth
Output directory: ../deconv_results/DATA18/
Device: cpu
Weight threshold: 0.0001
Loading pretrained VAE Encoder...
   VAE architecture: 1655 -> 256
   Output type: zinb
   Architecture: Dual Decoder (SC/ST-specific)
   ✓ Loaded 9999 cell cluster labels from checkpoint
Loading cluster data from: ../deconv_results/DATA18/final_vae_cluster_data.npz
   Cluster prototypes: torch.Size([74, 256])
   Cluster expressions (marker): torch.Size([74, 1655])
   Cluster expressions (all genes): 74 × 19886
   Loaded celltype mapping: 74 clusters
   Average cell counts: 11809.2
Loaded all genes list: 19886 genes
VAE Encoder loaded: 1655 -> 256
Cell type clusters: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '4', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '5',

GAT Training: 100%|██████████| 250/250 [09:57<00:00,  2.39s/epoch, Total=17.3821, MSE=30.4307, Spot_Cosine=0.2058, Gene_Cosine=0.1366, Pearson=0.3879, Gene_Pearson=0.4922, P_pat=51, M_pat=13, C_pat=51]


Evaluating model results...
Applying weight threshold: 0.0001
   Non-zero elements: 15000 -> 13242 (17.9%)
Saving deconvolution results...
Generating deconvolution expression matrices...
   Reconstructing all gene expression...
   Saved: ../deconv_results/DATA18//Spatial_reconstructed_all_genes.csv
   Cell type composition...
   Found duplicate celltype names: ['DG', 'Sst', 'L4_5 IT CTX', 'Vip', 'Oligo', 'Pvalb', 'L2_3 IT PPP', 'L2_3 IT CTX', 'L6 CT CTX', 'L5 PT CTX', 'L6 IT CTX', 'Lamp5']. Merging corresponding cluster columns by summing weights.
   Columns before: 74, after merge: 34
   Saved cell composition (celltype): ../deconv_results/DATA18//Spatial_cell_composition.csv
   Saved cluster composition: ../deconv_results/DATA18//Spatial_cluster_composition.csv
   Using 1655 marker genes for reconstruction quality (consistent with training objective)
   Cosine similarities saved: ../deconv_results/DATA18//Spatial_spot_cosine_similarity.csv

Plotting reconstruction quality curve...
  